In [2]:
import cv2
import glob
import numpy as np
from tqdm import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from collections import Counter
import random
import imageio
from IPython.display import Image
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from ultralytics import YOLO
import os

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\73671\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
MAX_SEQUENCE_LENGTH = 40
IMG_SIZE = 299
NUM_FEATURES = 1536

In [4]:
# 图像中心裁剪函数
def crop_center_square(img):
    h, w = img.shape[:2]
    square_w = min(h, w)

    start_x = w // 2 - square_w // 2
    end_x = start_x + square_w

    start_y = h // 2 - square_w // 2
    end_y = start_y + square_w

    result = img[start_y:end_y, start_x:end_x]
    return result

# 肤色检测和形态学处理函数
def preprocess_hand(frame):
    # 转换为 YCrCb 颜色空间
    ycrcb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2YCrCb)
    # 定义肤色范围
    lower_skin = np.array([0, 133, 77], dtype=np.uint8)
    upper_skin = np.array([255, 173, 127], dtype=np.uint8)
    # 肤色分割
    skin_mask = cv2.inRange(ycrcb_frame, lower_skin, upper_skin)

    # 形态学处理：开运算去噪，闭运算填充空洞
    kernel = np.ones((3, 3), np.uint8)
    skin_mask = cv2.morphologyEx(skin_mask, cv2.MORPH_OPEN, kernel)
    skin_mask = cv2.morphologyEx(skin_mask, cv2.MORPH_CLOSE, kernel)

    # 将肤色区域与原图像融合
    result = cv2.bitwise_and(frame, frame, mask=skin_mask)
    return result

In [6]:
# 图像中心裁剪函数
def crop_center_square(img):
    h, w = img.shape[:2]
    square_w = min(h, w)

    start_x = w // 2 - square_w // 2
    end_x = start_x + square_w

    start_y = h // 2 - square_w // 2
    end_y = start_y + square_w

    result = img[start_y:end_y, start_x:end_x]
    return result

# 肤色检测和形态学处理函数
def preprocess_hand(frame):
    # 转换为 YCrCb 颜色空间
    ycrcb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2YCrCb)
    # 定义肤色范围
    lower_skin = np.array([0, 133, 77], dtype=np.uint8)
    upper_skin = np.array([255, 173, 127], dtype=np.uint8)
    # 肤色分割
    skin_mask = cv2.inRange(ycrcb_frame, lower_skin, upper_skin)

    # 形态学处理：开运算去噪，闭运算填充空洞
    kernel = np.ones((3, 3), np.uint8)
    skin_mask = cv2.morphologyEx(skin_mask, cv2.MORPH_OPEN, kernel)
    skin_mask = cv2.morphologyEx(skin_mask, cv2.MORPH_CLOSE, kernel)

    # 将肤色区域与原图像融合
    result = cv2.bitwise_and(frame, frame, mask=skin_mask)
    return result

In [59]:
# 视频抽帧预处理函数
def load_video(file_name):
    cap = cv2.VideoCapture(file_name)
    if not cap.isOpened():
        print(f"Error opening video file: {file_name}")
        return []
    # 每隔多少帧抽取一次
    frame_interval = 4
    frames = []
    count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Video reading ended.")
            break
        if frame is None:
            print("Empty frame read. Skipping...")
            continue
        # 每隔frame_interval帧保存一次
        if count % frame_interval == 0:
            # 中心裁剪
            frame = crop_center_square(frame)

            try:
                # 使用 YOLO 进行手部检测
                results = yolo_model(frame)[0]
                hand_boxes = []
                for result in results.boxes.data.tolist():
                    x1, y1, x2, y2, score, class_id = result
                    # 这里假设类别ID 0 代表手，你需要根据实际的 YOLO 模型输出调整
                    if class_id == 0:
                        hand_boxes.append([int(x1), int(y1), int(x2), int(y2)])

                if hand_boxes:
                    # 这里简单取第一个检测到的手部区域，你可以根据需求调整
                    x1, y1, x2, y2 = hand_boxes[0]
                    frame = frame[y1:y2, x1:x2]
            except:
                print("Error in YOLO model processing. Skipping this frame...")
                continue

            # 肤色检测和形态学处理
            frame = preprocess_hand(frame)
            # 缩放
            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
            # BGR -> RGB  [0,1,2] -> [2,1,0]
            frame = frame[:, :, [2, 1, 0]]

            assert frame.ndim == 3, f"Frame should be 3D array, but got shape {frame.shape}"
            frames.append(frame)
            print(f"Added frame to frames list, frames list length: {len(frames)}")
        count += 1

    cap.release()  # 释放视频资源
    frames = np.array(frames)
    assert frames.ndim == 4, f"Frames should be 4D array, but got shape {frames.shape}"
    print(f"frames after conversion to array, type: {type(frames)}, shape: {frames.shape}")
    return frames

In [60]:
# 提取视频图像特征函数
def load_data(video, label):
    frames = load_video(video)
    counts = len(frames)
    # 如果帧数小于MAX_SEQUENCE_LENGTH
    if counts < MAX_SEQUENCE_LENGTH:
        # 补白
        diff = MAX_SEQUENCE_LENGTH - counts
        # 创建全0的numpy数组，确保维度与frames一致
        padding = np.zeros((diff, *frames.shape[1:]))

        assert frames.ndim == 4, f"Frames should be 4D array, but got shape {frames.shape}"
        assert padding.ndim == 4, f"Padding should be 4D array, but got shape {padding.shape}"

        # 数组拼接
        frames = np.concatenate((frames, padding))
    # 获取前MAX_SEQUENCE_LENGTH帧画面
    frames = frames[:MAX_SEQUENCE_LENGTH, :]
    # 批量提取特征
    video_feature = feature_extractor.predict(frames)
    return np.array([video_feature]), np.array([label])

In [61]:
# 提取视频图像特征函数
def load_data(video, label):
    frames = load_video(video)
    counts = len(frames)
    # 如果帧数小于MAX_SEQUENCE_LENGTH
    if counts < MAX_SEQUENCE_LENGTH:
        # 补白
        diff = MAX_SEQUENCE_LENGTH - counts
        # 确保 frames 是 numpy 数组
        if not isinstance(frames, np.ndarray):
            frames = np.array(frames)
        # 创建全0的numpy数组，确保维度与frames一致
        padding = np.zeros((diff, *frames.shape[1:]))

        print(f"Before concatenate, frames shape: {frames.shape}, padding shape: {padding.shape}")
        # 数组拼接
        frames = np.concatenate((frames, padding))
        print(f"After concatenate, frames shape: {frames.shape}")

    print(f"Before slicing, frames shape: {frames.shape}")
    # 获取前MAX_SEQUENCE_LENGTH帧画面
    frames = frames[:MAX_SEQUENCE_LENGTH, :]
    print(f"After slicing, frames shape: {frames.shape}")
    # 批量提取特征
    video_feature = feature_extractor.predict(frames)
    return np.array([video_feature]), np.array([label])

In [62]:
# 位置编码层类
class PositionalEmbedding(layers.Layer):
    def __init__(self, seq_length, output_dim):
        super().__init__()
        # 构造从0~MAX_SEQUENCE_LENGTH的列表
        self.positions = tf.range(0, limit=MAX_SEQUENCE_LENGTH)
        self.positional_embedding = layers.Embedding(input_dim=seq_length, output_dim=output_dim)

    def call(self, x):
        # 位置编码
        positions_embedding = self.positional_embedding(self.positions)
        # 输入相加
        return x + positions_embedding

# 编码器层类
class TransformerEncoder(layers.Layer):
    def __init__(self, num_heads, embed_dim):
        super().__init__()
        self.p_embedding = PositionalEmbedding(MAX_SEQUENCE_LENGTH, NUM_FEATURES)
        self.attention = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.1)
        self.layernorm = layers.LayerNormalization()

    def call(self, x):
        # positional embedding
        positional_embedding = self.p_embedding(x)
        # self attention
        attention_out = self.attention(
            query=positional_embedding,
            value=positional_embedding,
            key=positional_embedding,
            attention_mask=None
        )
        # layer norm with residual connection
        output = self.layernorm(positional_embedding + attention_out)
        return output

# 视频分类模型函数
def video_cls_model(class_vocab):
    # 类别数量
    classes_num = len(class_vocab)
    # 定义模型
    model = keras.Sequential([
        layers.InputLayer(input_shape=(MAX_SEQUENCE_LENGTH, NUM_FEATURES)),
        TransformerEncoder(2, NUM_FEATURES),
        layers.GlobalMaxPooling1D(),
        layers.Dropout(0.1),
        layers.Dense(classes_num, activation="softmax")
    ])
    # 编译模型
    model.compile(optimizer=keras.optimizers.Adam(1e-5),
                  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
                  metrics=['accuracy']
                  )
    return model

In [63]:
if __name__ == '__main__':
    print('Tensorflow version: {}'.format(tf.__version__))
    print('GPU available: {}'.format(tf.config.list_physical_devices('GPU')))

    # 初始化 YOLO 模型
    yolo_model = YOLO('yolov8n.pt')

    # 获取当前工作目录
    current_dir = os.getcwd()
    # 视频文件所在文件夹路径
    cv_folder_path = os.path.join(current_dir, 'cv')
    # 列出 cv 文件夹下所有视频文件
    videos = [f for f in os.listdir(cv_folder_path) if f.endswith('.mp4')]
    if videos:
        # 只取第一个视频
        first_video = videos[0]
        print(f"Video path: {first_video}")  # 添加这行代码检查视频路径
        label = int(first_video.split('_')[-2])

        feature_extractor = get_feature_extractor()
        video_features, classes = load_data(first_video, label)

        # 模型实例化
        model = video_cls_model(np.unique([classes]))
        # 打印模型结构
        model.summary()

        # 由于只有一个样本，这里仅作示例，实际训练需要更多数据
        # 保存检查点
        checkpoint = ModelCheckpoint(filepath='best.h5', monitor='loss', save_weights_only=True, save_best_only=True,
                                     verbose=1, mode='min')
        # 提前终止
        earlyStopping = EarlyStopping(monitor='loss', patience=50, mode='min', baseline=None)
        # 减少learning rate
        rlp = ReduceLROnPlateau(monitor='loss', factor=0.7, patience=30, min_lr=1e-15, mode='min', verbose=1)

        # 开始训练（这里仅作演示，单个样本训练意义不大）
        history = model.fit(video_features, classes,
                            epochs=10,
                            callbacks=[checkpoint, earlyStopping, rlp])
        # 绘制结果
        plt.plot(history.epoch, history.history['loss'], 'r', label='loss')
        plt.plot(history.epoch, history.history['val_loss'], 'g--', label='val_loss')
        plt.title('VIT Model')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()

        plt.plot(history.epoch, history.history['accuracy'], 'r', label='acc')
        plt.plot(history.epoch, history.history['val_accuracy'], 'g--', label='val_acc')
        plt.title('VIT Model')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()

        # 加载训练最优权重
        model.load_weights('best.h5')

        # 模型评估（这里仅对单个样本评估，实际需要更多样本）
        loss, accuracy = model.evaluate(video_features, classes)
        print(f"Loss: {loss}, Accuracy: {accuracy}")

        # 手势识别
        # 类别标签
        label_to_name = {0: '无效手势', 1: '上滑', 2: '下滑', 3: '左滑', 4: '右滑', 5: '打开', 6: '关闭', 7: '放大', 8: '缩小'}
        prob = model.predict(video_features)[0]
        predicted_class_index = np.argmax(prob)
        print(f"预测类别: {label_to_name[predicted_class_index]}, 概率: {prob[predicted_class_index]}")
    else:
        print("没有找到视频文件")

Tensorflow version: 2.13.0
GPU available: []
Video path: man_001_0_1.mp4
Error opening video file: man_001_0_1.mp4
Before concatenate, frames shape: (0,), padding shape: (40,)
After concatenate, frames shape: (40,)
Before slicing, frames shape: (40,)


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed